## Import and load

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from collections import Counter
from tqdm import tqdm

# Check for MPS support because macbook
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

glove_file = 'glove.6B/glove.6B.100d.txt'
if not os.path.exists(glove_file):
    raise FileNotFoundError(f"{glove_file} not found. Please download it from https://nlp.stanford.edu/projects/glove/ and place it in your working directory.")

df_train = pd.read_csv('train_preprocessed.csv')
df_test = pd.read_csv('test_preprocessed.csv')

# Ensure there are no NaN values in the text column
df_train['cleaned_text'] = df_train['cleaned_text'].fillna("")
df_test['cleaned_text'] = df_test['cleaned_text'].fillna("")

X_train = df_train['cleaned_text'].values
y_train = df_train['Bias'].values
X_test = df_test['cleaned_text'].values
y_test = df_test['Bias'].values

Using device: mps


## Preprocessing

In [2]:
# -----------------------
# 2. Encode Labels
# -----------------------
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
num_classes = len(le.classes_)
print("Number of classes:", num_classes)

Number of classes: 5


## Global variables and functions for both GloVe and Doc2Vec

In [3]:
# Simple whitespace tokenization (can substitute with a more robust tokenizer if needed)

def tokenize(text):
    return text.lower().split()

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

batch_size = 32
embedding_dim = 100  # Using GloVe 100d embeddings
num_epochs = 10

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=1e-3) ##model has not been defined here yet

In [4]:

# -----------------------
# 3. Tokenization and Padding
# -----------------------

# Build vocabulary from training texts (limit vocabulary size)
max_words = 20000
counter = Counter()
for text in X_train:
    tokens = tokenize(text)
    counter.update(tokens)

# Reserve indices for PAD and OOV tokens
most_common = counter.most_common(max_words - 2)
word2idx = {"<PAD>": 0, "<OOV>": 1}
for word, count in most_common:
    word2idx[word] = len(word2idx)
vocab_size = len(word2idx)
print("Vocabulary size:", vocab_size)

# Convert texts to sequences of word indices
def text_to_sequence(text, word2idx):
    tokens = tokenize(text)
    return [word2idx.get(token, word2idx["<OOV>"]) for token in tokens]

X_train_seq = [text_to_sequence(text, word2idx) for text in X_train]
X_test_seq = [text_to_sequence(text, word2idx) for text in X_test]

# Pad sequences to a fixed length
max_seq_length = 100
def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        return seq + [word2idx["<PAD>"]] * (max_len - len(seq))
    else:
        return seq[:max_len]

X_train_pad = [pad_sequence(seq, max_seq_length) for seq in X_train_seq]
X_test_pad = [pad_sequence(seq, max_seq_length) for seq in X_test_seq]

# Convert lists to PyTorch tensors
X_train_tensor = torch.LongTensor(X_train_pad)
y_train_tensor = torch.LongTensor(y_train_encoded)
X_test_tensor = torch.LongTensor(X_test_pad)
y_test_tensor = torch.LongTensor(y_test_encoded)

Vocabulary size: 20000


## GloVe 

In [5]:
# -----------------------
# 5. Load GloVe Embeddings and Build Embedding Matrix
# -----------------------
train_dataset = TextDataset(X_train_tensor, y_train_tensor)
test_dataset = TextDataset(X_test_tensor, y_test_tensor)


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


embeddings_index = {}
with open(glove_file, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector
print("Loaded %d word vectors from GloVe." % len(embeddings_index))

# Build embedding matrix: for each word in our vocabulary, use the GloVe vector if available
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in word2idx.items():
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

# Convert embedding matrix to PyTorch tensor
embedding_matrix_tensor = torch.FloatTensor(embedding_matrix)

Loaded 400000 word vectors from GloVe.


In [6]:
# -----------------------
# 6. Define the LSTM Model in PyTorch
# -----------------------
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix, num_layers=1, dropout=0.5):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight = nn.Parameter(embedding_matrix)
        self.embedding.weight.requires_grad = False  # Freeze embeddings; set True to fine-tune
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        embedded = self.embedding(x)  # [batch, seq_len, embedding_dim]
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Use the last hidden state (from the last layer)
        hidden = self.dropout(hidden[-1])
        out = self.fc(hidden)
        return out

hidden_dim = 128
num_layers = 1
dropout = 0.5
model = LSTMClassifier(vocab_size=vocab_size,
                       embedding_dim=embedding_dim,
                       hidden_dim=hidden_dim,
                       output_dim=num_classes,
                       embedding_matrix=embedding_matrix_tensor,
                       num_layers=num_layers,
                       dropout=dropout)
model = model.to(device)
print(model)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

LSTMClassifier(
  (embedding): Embedding(20000, 100)
  (lstm): LSTM(100, 128, batch_first=True, dropout=0.5)
  (fc): Linear(in_features=128, out_features=5, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


/Users/foongming/.pyenv/versions/3.12.3/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


In [7]:
# -----------------------
# 7. Train the Model
# -----------------------

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    epoch_correct = 0
    total = 0
    for texts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * texts.size(0)
        _, predicted = torch.max(outputs, 1)
        epoch_correct += (predicted == labels).sum().item()
        total += labels.size(0)
    epoch_acc = epoch_correct / total
    print(f"Epoch {epoch+1} Loss: {epoch_loss/total:.4f} Acc: {epoch_acc:.4f}")

Epoch 1/10: 100%|██████████| 56/56 [00:01<00:00, 46.92it/s]


Epoch 1 Loss: 1.4039 Acc: 0.4829


Epoch 2/10: 100%|██████████| 56/56 [00:00<00:00, 63.29it/s]


Epoch 2 Loss: 1.3296 Acc: 0.5171


Epoch 3/10: 100%|██████████| 56/56 [00:00<00:00, 61.26it/s]


...

Epoch 8 Loss: 1.1608 Acc: 0.5509


Epoch 9/10: 100%|██████████| 56/56 [00:01<00:00, 55.30it/s]


Epoch 9 Loss: 1.0959 Acc: 0.5942


Epoch 10/10: 100%|██████████| 56/56 [00:00<00:00, 65.05it/s]

Epoch 10 Loss: 1.0275 Acc: 0.5992


In [8]:
# -----------------------
# 8. Evaluate the Model on the Test Set
# -----------------------
model.eval()
test_loss = 0
correct = 0
total = 0
all_preds = []
all_labels = []
with torch.no_grad():
    for texts, labels in test_loader:
        texts, labels = texts.to(device), labels.to(device)
        outputs = model(texts)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * texts.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= total
test_acc = correct / total
print(f"Test Loss: {test_loss:.4f} Test Accuracy: {test_acc:.4f}")

# Print classification report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

Test Loss: 1.4712 Test Accuracy: 0.4719
Classification Report:
              precision    recall  f1-score   support

      center       0.13      0.05      0.07        40
   lean left       0.08      0.03      0.05        61
  lean right       0.00      0.00      0.00        37
        left       0.54      0.87      0.67       230
       right       0.24      0.09      0.13        77

    accuracy                           0.47       445
   macro avg       0.20      0.21      0.18       445
weighted avg       0.34      0.47      0.38       445



## Doc2Vec

In [10]:
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

# Load the same data as in the LSTM notebook
df_train = pd.read_csv('train_preprocessed.csv')
df_test = pd.read_csv('test_preprocessed.csv')

# Ensure there are no NaN values in the text column
df_train['cleaned_text'] = df_train['cleaned_text'].fillna("")
df_test['cleaned_text'] = df_test['cleaned_text'].fillna("")

X_train = df_train['cleaned_text'].values
y_train = df_train['Bias'].values
X_test = df_test['cleaned_text'].values
y_test = df_test['Bias'].values

# -----------------------
# Doc2Vec to LSTM Implementation with Debugging
# -----------------------

# First, check for empty texts or texts that tokenize to empty lists
def tokenize(text):
    return text.lower().split()

# Check for problematic samples
empty_count = 0
for text in X_train:
    if len(tokenize(text)) == 0:
        empty_count += 1
print(f"Found {empty_count} texts that tokenize to empty lists in training data")

empty_count = 0
for text in X_test:
    if len(tokenize(text)) == 0:
        empty_count += 1
print(f"Found {empty_count} texts that tokenize to empty lists in test data")

# Add a default token for truly empty texts
def safe_tokenize(text):
    tokens = tokenize(text)
    if len(tokens) == 0:
        # Use a special token for empty texts
        return ["<pad>"]
    return tokens

# Prepare tagged documents for Doc2Vec training
print("Preparing documents for Doc2Vec training...")
train_documents = [TaggedDocument(safe_tokenize(text), [i]) for i, text in enumerate(X_train)]

# Train the Doc2Vec model
print("Training Doc2Vec model...")
vector_size = 100  # Same dimension as GloVe for comparison
doc2vec_model = Doc2Vec(
    documents=train_documents,
    vector_size=vector_size,
    window=5,
    min_count=1,  # Reduced to keep rare words
    workers=4,
    epochs=20,
    dm=1  # Use distributed memory
)

# Define a safer dataset class
class Doc2VecSeqDataset(Dataset):
    def __init__(self, texts, labels, doc2vec_model, max_length=None):
        self.texts = texts
        self.labels = labels
        self.doc2vec_model = doc2vec_model
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = safe_tokenize(text)
        
        # If max_length is specified, handle it
        if self.max_length is not None and len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]
        
        # Get vector for each token from Doc2Vec model
        token_vectors = []
        for token in tokens:
            if token in self.doc2vec_model.wv:
                token_vectors.append(self.doc2vec_model.wv[token])
            else:
                # For OOV tokens, use random vector
                token_vectors.append(np.random.normal(scale=0.6, size=(vector_size,)))
        
        # Ensure we have at least one vector (should never be empty due to safe_tokenize)
        if len(token_vectors) == 0:
            print(f"Warning: No vectors for text at index {idx}: '{text}'")
            # Add a default vector
            token_vectors.append(np.zeros(vector_size))
        
        # Convert to tensor
        token_vectors = torch.FloatTensor(token_vectors)
        
        # Get the sequence length
        seq_length = len(token_vectors)
        
        return {
            'token_vectors': token_vectors,
            'seq_length': seq_length,
            'label': torch.LongTensor([self.labels[idx]])[0]
        }

# Create a safer collate function
def collate_fn(batch):
    # Get sequence lengths (should all be > 0 now)
    seq_lengths = [x['seq_length'] for x in batch]
    
    # Double-check there are no zero lengths
    if min(seq_lengths) == 0:
        print("Warning: Zero-length sequence found in batch")
        # Fix any zero lengths to be 1
        seq_lengths = [max(1, length) for length in seq_lengths]
    
    # Sort batch by sequence length (descending)
    sorted_indices = sorted(range(len(seq_lengths)), key=lambda i: seq_lengths[i], reverse=True)
    sorted_batch = [batch[i] for i in sorted_indices]
    sorted_seq_lengths = [seq_lengths[i] for i in sorted_indices]
    
    # Get max sequence length in this batch
    max_seq_len = max(sorted_seq_lengths)
    
    # Prepare padded sequences and labels
    batch_size = len(sorted_batch)
    padded_vectors = torch.zeros(batch_size, max_seq_len, vector_size)
    labels = torch.zeros(batch_size, dtype=torch.long)
    
    # Fill in the data
    for i, item in enumerate(sorted_batch):
        seq_len = item['seq_length']
        padded_vectors[i, :seq_len] = item['token_vectors']
        labels[i] = item['label']
    
    return {
        'token_vectors': padded_vectors,
        'seq_lengths': torch.LongTensor(sorted_seq_lengths),
        'labels': labels
    }

# Create datasets and dataloaders
train_dataset = Doc2VecSeqDataset(X_train, y_train_encoded, doc2vec_model)
test_dataset = Doc2VecSeqDataset(X_test, y_test_encoded, doc2vec_model)

batch_size = 32
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn
)

# Define the LSTM model that handles variable-length sequences
class VariableLengthLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, dropout=0.5):
        super(VariableLengthLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # LSTM layer
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x, seq_lengths):
        # Pack padded sequence
        packed_input = nn.utils.rnn.pack_padded_sequence(
            x, seq_lengths.cpu(), batch_first=True, enforce_sorted=True
        )
        
        # Forward pass through LSTM
        packed_output, (hidden, _) = self.lstm(packed_input)
        
        # Get the last hidden state
        hidden = hidden[-1]
        hidden = self.dropout(hidden)
        
        # Pass through linear layer
        output = self.fc(hidden)
        
        return output

# Initialize the model
hidden_dim = 128  # Same as original LSTM
doc2vec_lstm = VariableLengthLSTM(
    input_dim=vector_size,
    hidden_dim=hidden_dim,
    output_dim=num_classes,
    num_layers=1,
    dropout=0.5
).to(device)

print(doc2vec_lstm)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(doc2vec_lstm.parameters(), lr=1e-3)

# Train the model
num_epochs = 10
print("Training Variable-Length LSTM with Doc2Vec features...")
for epoch in range(num_epochs):
    doc2vec_lstm.train()
    epoch_loss = 0
    epoch_correct = 0
    total = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        token_vectors = batch['token_vectors'].to(device)
        seq_lengths = batch['seq_lengths']
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = doc2vec_lstm(token_vectors, seq_lengths)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        epoch_correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    epoch_acc = epoch_correct / total
    print(f"Epoch {epoch+1} Loss: {epoch_loss/total:.4f} Acc: {epoch_acc:.4f}")

# Evaluate the model
doc2vec_lstm.eval()
test_loss = 0
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        token_vectors = batch['token_vectors'].to(device)
        seq_lengths = batch['seq_lengths']
        labels = batch['labels'].to(device)
        
        outputs = doc2vec_lstm(token_vectors, seq_lengths)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= total
test_acc = correct / total
print(f"Variable-Length LSTM with Doc2Vec Test Loss: {test_loss:.4f} Test Accuracy: {test_acc:.4f}")

# Print classification report
print("Variable-Length LSTM with Doc2Vec Classification Report:")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

Found 16 texts that tokenize to empty lists in training data
Found 3 texts that tokenize to empty lists in test data
Preparing documents for Doc2Vec training...
Training Doc2Vec model...
VariableLengthLSTM(
  (lstm): LSTM(100, 128, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)
Training Variable-Length LSTM with Doc2Vec features...


Epoch 1/10: 100%|██████████| 56/56 [01:54<00:00,  2.04s/it]


Epoch 1 Loss: 1.2344 Acc: 0.5273


Epoch 2/10: 100%|██████████| 56/56 [01:37<00:00,  1.74s/it]


Epoch 2 Loss: 0.9612 Acc: 0.6464


...

Epoch 8 Loss: 0.6089 Acc: 0.7746


Epoch 9/10: 100%|██████████| 56/56 [01:34<00:00,  1.68s/it]


Epoch 9 Loss: 0.5441 Acc: 0.8128


Epoch 10/10: 100%|██████████| 56/56 [01:37<00:00,  1.74s/it]


Epoch 10 Loss: 0.4899 Acc: 0.8286
Variable-Length LSTM with Doc2Vec Test Loss: 0.8852 Test Accuracy: 0.6809
Variable-Length LSTM with Doc2Vec Classification Report:
              precision    recall  f1-score   support

      center       0.58      0.45      0.51        40
   lean left       0.49      0.56      0.52        61
  lean right       0.29      0.16      0.21        37
        left       0.74      0.83      0.79       230
       right       0.83      0.69      0.75        77

    accuracy                           0.68       445
   macro avg       0.58      0.54      0.55       445
weighted avg       0.67      0.68      0.67       445



In [11]:

# -----------------------
# Model Comparison Summary
# -----------------------

print("\nModel Comparison Summary:")
print("1. LSTM with GloVe word embeddings: [accuracy from your original model]")
print(f"2. LSTM with Doc2Vec word embeddings: {lstm_doc2vec_acc:.4f}")
print(f"3. Simple classifier with Doc2Vec document embeddings: {pure_doc2vec_acc:.4f}")


Model Comparison Summary:
1. LSTM with GloVe word embeddings: [accuracy from your original model]


NameError: name 'lstm_doc2vec_acc' is not defined